In [6]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.entities import ManagedOnlineEndpoint, ManagedOnlineDeployment, Model
import datetime

# Connect to the workspace
ml_client = MLClient.from_config(credential=DefaultAzureCredential())

print("Successfully connected to Azure ML Workspace")

Successfully connected to Azure ML Workspace


In [2]:
# Generate a unique name
timestamp = datetime.datetime.now().strftime("%m%d%H%M")
endpoint_name = f"library-endpoint-{timestamp}"

endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    description="Online endpoint for library occupancy forecasting",
    auth_mode="key",
)

# This command sends the request to Azure to build the infrastructure
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

print(f"Endpoint created: {endpoint_name}")

/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.8.1) and child packages mlflow-skinny (3.5.0) are different. This may lead to unexpected behavior. Please install the same version of all MLflow packages.
  mlflow.mismatch._check_version_mismatch()


Endpoint created: library-endpoint-04101503


In [11]:
%%writefile score.py
import json
import joblib
import numpy as np
import os

def init():
    global model
    # This path is where Azure puts your model files
    model_path = os.path.join(os.getenv("AZUREML_MODEL_DIR"), "model/model.pkl")
    model = joblib.load(model_path)

def run(raw_data):
    try:
        data = json.loads(raw_data)["data"]
        data = np.array(data)
        result = model.predict(data)
        return result.tolist()
    except Exception as e:
        return str(e)

Writing score.py


In [1]:
import numpy, sklearn, pandas
print(f"numpy=={numpy.__version__}")
print(f"scikit-learn=={sklearn.__version__}")
print(f"pandas=={pandas.__version__}")

numpy==1.23.5
scikit-learn==1.7.2
pandas==1.5.3


In [7]:
from azure.ai.ml.entities import ManagedOnlineDeployment, Environment

# 1. Define the environment with your pinned versions
env = Environment(
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04",
    conda_file="conda.yml",
    name="pinned-inference-env"
)

# 2. Configure the deployment
v11_deployment = ManagedOnlineDeployment(
    name="final-v11",
    endpoint_name="library-endpoint-04101503", # Verify this matches your endpoint tab
    model=ml_client.models.get(name="library_occupancy_rf_model", version="2"),
    environment=env,
    instance_type="Standard_DS3_v2",
    instance_count=1,
)

# 3. Start the deployment
print("🚀 Launching v11 with pinned versions. This confirms deployment configuration...")
ml_client.online_deployments.begin_create_or_update(v11_deployment).result()

# 4. Route 100% of traffic to this new deployment
endpoint = ml_client.online_endpoints.get(name="library-endpoint-04101503")
endpoint.traffic = {"final-v11": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

print("✅ Deployment v11 complete. Check the 'Endpoints' tab for Healthy status.")

🚀 Launching v11 with pinned versions. This confirms deployment configuration...
......

KeyboardInterrupt: 

In [8]:
# Replace 'final-v11' with the name of your deployment if different
logs = ml_client.online_deployments.get_logs(
    name="final-v11", 
    endpoint_name="library-endpoint-04101503", 
    lines=50
)
print(logs)

Instance status:
SystemSetup: Succeeded
UserContainerImagePull: Succeeded
ModelDownload: Succeeded
UserContainerStart: InProgress

Container events:
Kind: Pod, Name: Downloading, Type: Normal, Time: 2026-04-10T20:22:22.434909Z, Message: Start downloading models
Kind: Pod, Name: Pulling, Type: Normal, Time: 2026-04-10T20:22:22.722718Z, Message: Start pulling container image
Kind: Pod, Name: Pulled, Type: Normal, Time: 2026-04-10T20:23:12.969474Z, Message: Container image is pulled successfully
Kind: Pod, Name: Downloaded, Type: Normal, Time: 2026-04-10T20:23:12.969474Z, Message: Models are downloaded successfully
Kind: Pod, Name: Created, Type: Normal, Time: 2026-04-10T20:23:13.004834Z, Message: Created container inference-server
Kind: Pod, Name: Started, Type: Normal, Time: 2026-04-10T20:23:13.047954Z, Message: Started container inference-server

Container logs:
pydantic==2.12.5
pydantic_core==2.41.5
Pygments==2.20.0
PyJWT==2.12.1
pyparsing==3.3.2
python-dateutil==2.9.0.post0
python-do

In [9]:
%%writefile conda.yml
name: library-env
channels:
  - conda-forge
  - defaults
dependencies:
  - python=3.10
  - pip:
    - numpy==1.23.5
    - scikit-learn==1.7.2
    - pandas==1.5.3
    - mlflow
    - azureml-mlflow
    - cloudpickle
    - azureml-inference-server-http  # <--- Added this to fix the Log Error

Overwriting conda.yml


In [10]:
from azure.ai.ml.entities import ManagedOnlineDeployment, Environment

# Define the environment with the missing package included
env = Environment(
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04",
    conda_file="conda.yml",
    name="final-fix-env"
)

v12_deployment = ManagedOnlineDeployment(
    name="final-v12",
    endpoint_name="library-endpoint-04101503",
    model=ml_client.models.get(name="library_occupancy_rf_model", version="2"),
    environment=env,
    instance_type="Standard_DS3_v2",
    instance_count=1,
)

print("🚀 Launching v12. This iteration targets the 'Exit code 100' dependency error.")
ml_client.online_deployments.begin_create_or_update(v12_deployment).result()

# Route traffic
endpoint = ml_client.online_endpoints.get(name="library-endpoint-04101503")
endpoint.traffic = {"final-v12": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

print("✅ v12 Submitted. Check the Endpoints tab!")

🚀 Launching v12. This iteration targets the 'Exit code 100' dependency error.
................................................................................................................................................................................................

HttpResponseError: (BadArgument) User container has crashed or terminated. Please see troubleshooting guide, available here: https://aka.ms/oe-tsg#error-resourcenotready
Code: BadArgument
Message: User container has crashed or terminated. Please see troubleshooting guide, available here: https://aka.ms/oe-tsg#error-resourcenotready

In [11]:
# Final proof for Mr. Elyor: Checking the logs for the v12 fixed deployment
v12_logs = ml_client.online_deployments.get_logs(
    name="final-v12", 
    endpoint_name="library-endpoint-04101503", 
    lines=50
)

print(v12_logs)

Instance status:
SystemSetup: Succeeded
UserContainerImagePull: Succeeded
ModelDownload: Succeeded
UserContainerStart: InProgress

Container events:
Kind: Pod, Name: Pulling, Type: Normal, Time: 2026-04-10T20:42:33.717819Z, Message: Start pulling container image
Kind: Pod, Name: Downloading, Type: Normal, Time: 2026-04-10T20:42:33.815043Z, Message: Start downloading models
Kind: Pod, Name: Pulled, Type: Normal, Time: 2026-04-10T20:43:25.206506Z, Message: Container image is pulled successfully
Kind: Pod, Name: Downloaded, Type: Normal, Time: 2026-04-10T20:43:25.206506Z, Message: Models are downloaded successfully
Kind: Pod, Name: Created, Type: Normal, Time: 2026-04-10T20:43:25.239019Z, Message: Created container inference-server
Kind: Pod, Name: Started, Type: Normal, Time: 2026-04-10T20:43:25.285926Z, Message: Started container inference-server

Container logs:
2026-04-10 20:43:29,176 I [74] azmlinfsrv - Starting up app insights client
2026-04-10 20:43:29,177 E [74] azmlinfsrv - No su